In [121]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [122]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [123]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,-676351497 | INFO | 4034756 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc
0����,-676351495 | INFO | 4034756 - Setting seeds to: 0
!,-676351483 | INFO | 4034756 - REMOVAL TYPE SET AS edge
,-676351482 | INFO | 4034756 - Caching System: False.
8ya!�,-676351452 | INFO | 4034756 - Instantiating: torch_geometric.datasets.Planetoid
�#��,-676351425 | INFO | 4034756 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
��m��,-676351425 | INFO | 4034756 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Citeseer'}, 'preprocess': []}}}
8ya!�,-676351388 | INFO | 4034756 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage


�T6�,-676351335 | INFO | 4034756 - ['all', 'all_shuffled', '-']
8ya!�,-676351333 | INFO | 4034756 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
�Y��,-676351332 | INFO | 4034756 - ['all', 'all_shuffled', '-', 'train_0', 'test']
8ya!�,-676351331 | INFO | 4034756 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
��N��,-676351330 | INFO | 4034756 - ['all', 'all_shuffled', '-', 'train_0', 'test', 'validation', 'train']
8ya!�,-676351321 | INFO | 4034756 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
�~t,-676351287 | INFO | 4034756 - ['all', 'all_shuffled', '-', 'train_0', 'test', 'validation', 'train', 'forget', 'retain']
0�9��,-676351265 | INFO | 4034756 - ['all', 'all_shuffled', '-', 'train_0', 'test', 'validation', 'train', 'forget', 'retain']
�#��,-676351254 | INFO | 4034756 - Created Configurable: erasure.data.datasets.DatasetManager.DatasetManager


In [124]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

8ya!�,-676351151 | INFO | 4034756 - Instantiating: erasure.model.graphs.GCN.GCN


8ya!�,-676351142 | INFO | 4034756 - Instantiating: torch.optim.Adam
8ya!�,-676351141 | INFO | 4034756 - Instantiating: torch.nn.CrossEntropyLoss
graph has edges  torch.Size([2, 9104])
��t,-676351063 | INFO | 4034756 - epoch = 0 ---> loss = 1.7978	 accuracy = 0.1720
graph has edges  torch.Size([2, 9104])
��t,-676351006 | INFO | 4034756 - epoch = 1 ---> loss = 1.7470	 accuracy = 0.3599
graph has edges  torch.Size([2, 9104])
��t,-676350951 | INFO | 4034756 - epoch = 2 ---> loss = 1.6973	 accuracy = 0.5436
graph has edges  torch.Size([2, 9104])
��t,-676350893 | INFO | 4034756 - epoch = 3 ---> loss = 1.6504	 accuracy = 0.6200
graph has edges  torch.Size([2, 9104])
��t,-676350834 | INFO | 4034756 - epoch = 4 ---> loss = 1.5992	 accuracy = 0.6785
graph has edges  torch.Size([2, 9104])
��t,-676350768 | INFO | 4034756 - epoch = 5 ---> loss = 1.5530	 accuracy = 0.7027
graph has edges  torch.Size([2, 9104])
��t,-676350714 | INFO | 4034756 - epoch = 6 ---> loss = 1.5004	 accuracy = 0.7219
graph 

In [125]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [126]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [127]:
import torch
print(torch.version.cuda)

12.6


In [128]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.0005000000000000002
    maximize: False
    weight_decay: 0
)


In [129]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [130]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

�#��,-676347821 | INFO | 4034756 - Created Configurable: erasure.unlearners.graph_unlearners.FisherForgetting.FisherForgetting


/NFSHOME/adangelo/LinkInference/erasure/unlearners/graph_unlearners/GraphUnlearner.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels)


In [131]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [132]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [133]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [134]:
print(len( data_manager.partitions['forget']) )

1820


In [135]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

�#��,-676347417 | INFO | 4034756 - Created Configurable: erasure.evaluations.running.RunTime
0�7��,-676347415 | INFO | 4034756 - Function: <function accuracy_score at 0x7fbf8c7bb3a0>
�#��,-676347415 | INFO | 4034756 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
@�7��,-676347414 | INFO | 4034756 - Function: <function accuracy_score at 0x7fbf8c7bb3a0>
�#��,-676347413 | INFO | 4034756 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
0,-676347412 | INFO | 4034756 - Function: <function accuracy_score at 0x7fbf8c7bb3a0>
�#��,-676347411 | INFO | 4034756 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
�$��,-676347410 | INFO | 4034756 - Function: <function accuracy_score at 0x7fbf8c7bb3a0>
�#��,-676347409 | INFO | 4034756 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
�#��,-676347408 | INFO | 4034756 - Created Configurable: erasure.evaluations.measures.SaveValues
�#��,-676347408 | INF

Traceback (most recent call last):
  File "/NFSHOME/adangelo/LinkInference/erasure/evaluations/manager.py", line 23, in evaluate
    e = measure.process(e)
  File "/NFSHOME/adangelo/LinkInference/erasure/evaluations/running.py", line 55, in process
    super().process(e)
  File "/NFSHOME/adangelo/LinkInference/erasure/evaluations/running.py", line 24, in process
    e.unlearned_model = e.unlearner.unlearn()
  File "/NFSHOME/adangelo/LinkInference/erasure/core/unlearner.py", line 25, in unlearn
    new_model = self.__unlearn__()
  File "/NFSHOME/adangelo/LinkInference/erasure/unlearners/graph_unlearners/FisherForgetting.py", line 118, in __unlearn__
    self.compute_fisher_information(retain_set)
  File "/NFSHOME/adangelo/LinkInference/erasure/unlearners/graph_unlearners/FisherForgetting.py", line 48, in compute_fisher_information
    p.grad_acc += (self.labels == target).float() * p.grad.data
RuntimeError: The size of tensor a (3327) must match the size of tensor b (100) at non-singlet